In [2]:
import sys, json, time
from pathlib import Path
import numpy as np
import pandas as pd


raiz = Path.cwd()
while not (raiz / "corrector").is_dir() and raiz != raiz.parent:
    raiz = raiz.parent
sys.path.insert(0, str(raiz))
from corrector.datos import cargar_splits
from corrector.lenguaje import ModeloLenguaje
from corrector.metricas import cer_wer
from corrector.beam import beam_ctc, greedy_ctc
from corrector.beam import prefijos_de

data = raiz / "Modelo/notebooks/data"
df = pd.read_csv(data / "val_predictions.csv").fillna("")
lm = ModeloLenguaje(cargar_splits()["train"]["label"])


pre = prefijos_de(lm.vocab)
print(len(pre), "prefijos")

lp_val = np.load(data / "val_logprobs.npz")
voc = json.load(open(data / "vocab_modelo.json", encoding="utf-8"))
idx_to_char = {int(i): c for i, c in voc["idx_to_char"].items()}
blank = voc["blank_idx"]

print(len(df), "filas en el CSV |", len(lp_val.files), "clips con log_probs | vocab", len(idx_to_char))

47736 prefijos
340 filas en el CSV | 340 clips con log_probs | vocab 35


In [3]:
clave = lambda fila: f"{fila.video_id}__{fila.clip}"
iguales = sum(greedy_ctc(lp_val[clave(f)].astype(np.float32), idx_to_char, blank)
              == " ".join(f.prediction.split()) for f in df.itertuples())
print(f"{iguales} de {len(df)} iguales al CSV")

336 de 340 iguales al CSV


In [ ]:
from corrector.metricas import cer_wer

videos = sorted(df["video_id"].unique())
pm = df[~df["video_id"].isin(set(videos[: len(videos) * 3 // 5]))]
muestra = pm.sample(25, random_state=0)
refs = muestra["reference"].tolist()
mats = [lp_val[clave(f)].astype(np.float32) for f in muestra.itertuples()]
print("argmax: CER {:.1%} | WER {:.1%}".format(*cer_wer(refs, muestra["prediction"].tolist())))

filas = []
for alfa in [0.3, 0.6, 1.0]:
    for beta in [0.0, 2.0, 4.0]:
        t0 = time.time()
        hyp = [beam_ctc(m, idx_to_char, blank, lm=lm, alfa=alfa, beta=beta, prefijos=pre) for m in mats]
        cer, wer = cer_wer(refs, hyp)
        filas.append({"alfa": alfa, "beta": beta, "CER": round(cer * 100, 1),
                      "WER": round(wer * 100, 1), "seg": round(time.time() - t0)})
        print(filas[-1])

argmax: CER 68.5% | WER 101.2%
{'alfa': 0.3, 'beta': 0.0, 'CER': 76.3, 'WER': 99.7, 'seg': 2}
{'alfa': 0.3, 'beta': 2.0, 'CER': 68.3, 'WER': 97.7, 'seg': 2}
{'alfa': 0.3, 'beta': 4.0, 'CER': 67.0, 'WER': 152.5, 'seg': 3}
{'alfa': 0.6, 'beta': 0.0, 'CER': 84.5, 'WER': 100.0, 'seg': 2}
{'alfa': 0.6, 'beta': 2.0, 'CER': 79.7, 'WER': 98.8, 'seg': 2}
{'alfa': 0.6, 'beta': 4.0, 'CER': 71.4, 'WER': 99.7, 'seg': 2}
{'alfa': 1.0, 'beta': 0.0, 'CER': 89.4, 'WER': 100.0, 'seg': 1}
{'alfa': 1.0, 'beta': 2.0, 'CER': 88.3, 'WER': 100.0, 'seg': 1}
{'alfa': 1.0, 'beta': 4.0, 'CER': 82.0, 'WER': 96.8, 'seg': 1}


In [5]:
filas = []
for alfa in [0.1, 0.2, 0.3, 0.4]:
    for beta in [2.0, 2.5, 3.0, 3.5]:
        t0 = time.time()
        hyp = [beam_ctc(m, idx_to_char, blank, lm=lm, alfa=alfa, beta=beta, prefijos=pre, B=32)
               for m in mats]
        cer, wer = cer_wer(refs, hyp)
        filas.append({"alfa": alfa, "beta": beta, "CER": round(cer * 100, 1),
                      "WER": round(wer * 100, 1), "seg": round(time.time() - t0)})
        print(filas[-1])
tabla = pd.DataFrame(filas)
tabla["CER+WER"] = tabla["CER"] + tabla["WER"]
tabla.sort_values("CER+WER").head(5)

{'alfa': 0.1, 'beta': 2.0, 'CER': 66.0, 'WER': 154.2, 'seg': 6}
{'alfa': 0.1, 'beta': 2.5, 'CER': 65.8, 'WER': 172.0, 'seg': 5}
{'alfa': 0.1, 'beta': 3.0, 'CER': 66.1, 'WER': 188.6, 'seg': 6}
{'alfa': 0.1, 'beta': 3.5, 'CER': 68.7, 'WER': 199.7, 'seg': 5}
{'alfa': 0.2, 'beta': 2.0, 'CER': 66.3, 'WER': 117.2, 'seg': 5}
{'alfa': 0.2, 'beta': 2.5, 'CER': 66.2, 'WER': 136.2, 'seg': 5}
{'alfa': 0.2, 'beta': 3.0, 'CER': 66.8, 'WER': 155.7, 'seg': 5}
{'alfa': 0.2, 'beta': 3.5, 'CER': 66.2, 'WER': 174.3, 'seg': 5}
{'alfa': 0.3, 'beta': 2.0, 'CER': 67.1, 'WER': 100.0, 'seg': 4}
{'alfa': 0.3, 'beta': 2.5, 'CER': 67.2, 'WER': 107.9, 'seg': 4}
{'alfa': 0.3, 'beta': 3.0, 'CER': 66.2, 'WER': 123.9, 'seg': 4}
{'alfa': 0.3, 'beta': 3.5, 'CER': 66.3, 'WER': 141.4, 'seg': 5}
{'alfa': 0.4, 'beta': 2.0, 'CER': 69.5, 'WER': 99.1, 'seg': 4}
{'alfa': 0.4, 'beta': 2.5, 'CER': 68.5, 'WER': 98.8, 'seg': 4}
{'alfa': 0.4, 'beta': 3.0, 'CER': 66.2, 'WER': 102.0, 'seg': 4}
{'alfa': 0.4, 'beta': 3.5, 'CER': 66.9, 'W

,alfa,beta,CER,WER,seg,CER+WER
8,0.3,2.0,67.1,100.0,4,167.1
13,0.4,2.5,68.5,98.8,4,167.3
14,0.4,3.0,66.2,102.0,4,168.2
12,0.4,2.0,69.5,99.1,4,168.6
9,0.3,2.5,67.2,107.9,4,175.1


In [9]:
mejor = tabla.sort_values("CER+WER").iloc[0]
hyp = [beam_ctc(m, idx_to_char, blank, lm=lm, alfa=mejor.alfa, beta=mejor.beta, prefijos=pre, B=32)
       for m in mats[:5]]
for r, p, h in zip(refs, muestra["prediction"], hyp):
    print(f"\nREAL: {r}\nARGM: {p}\nBEAM: {h}")


REAL: los resultados no solo serían relevantes sino que también serían comunitarios y también tendrían un impacto en el medioambiente sumamente positivo
ARGM: eoado ooa easeeeameteeo o peao eamee e ve paso a mea mete o o pamee moeo
BEAM: o a dos o a la base samente o pesado esa mente se ve paso a mi mente o mames modo

REAL: el sida no más adelante conocí a una pareja
ARGM: aea oe peta acooe e ona paa
BEAM: a ti e peace o es a una pera

REAL: se espera que el consumo energético de toda la humanidad pueda llegar a ser un cincuenta mayor
ARGM: o etetemaseamoo e eamaeeoo peeete a matatee poe e se uo eeo ete meteo
BEAM: o ese tema seamos e ameno peace amante de poe aseo o esa metido

REAL: güey si no nos dices no nos damos cuenta no estás con una especie de vitrina y de cristal
ARGM: ao e teto etamo o peta emo a mea etataa
BEAM: a lo que eso es amo o pata o a peace

REAL: en ese momento yo no sólo no tenía ni los doscientos
ARGM: ee pometoo o e tatecooaa
BEAM: ese momento no o esa seco o 

In [10]:
frec = sorted(lm.uni, key=lm.uni.get, reverse=True)
for N in [3000, 6000, len(frec)]:
    pre_n = prefijos_de(frec[:N])
    hyp = [beam_ctc(m, idx_to_char, blank, lm=lm, alfa=0.4, beta=3.0, prefijos=pre_n, B=32) for m in mats]
    print(f"N={N}: CER {cer_wer(refs, hyp)[0]:.1%} | WER {cer_wer(refs, hyp)[1]:.1%}")

N=3000: CER 67.6% | WER 107.0%
N=6000: CER 67.5% | WER 104.1%
N=16460: CER 66.2% | WER 102.0%


In [ ]:
mats_pm = [lp_val[clave(f)].astype(np.float32) for f in pm.itertuples()]
refs_pm = pm["reference"].tolist()
print("argmax: CER {:.1%} | WER {:.1%}".format(*cer_wer(refs_pm, pm["prediction"].tolist())))
for alfa, beta, N in [(0.4, 3.0, 6000), (0.3, 2.0, 6000)]:     # ajustá según lo que te dé
    pre_n = prefijos_de(frec[:N])
    hyp = [beam_ctc(m, idx_to_char, blank, lm=lm, alfa=alfa, beta=beta, prefijos=pre_n, B=32) for m in mats_pm]
    print(alfa, beta, N, "CER {:.1%} | WER {:.1%}".format(*cer_wer(refs_pm, hyp)))

argmax: CER 67.3% | WER 101.0%
0.4 3.0 6000 CER 66.7% | WER 104.2%
0.3 2.0 6000 CER 66.9% | WER 101.4%


In [12]:
for alfa, beta, B in [(0.4, 3.0, 32), (0.3, 2.0, 32), (0.4, 2.5, 32), (0.4, 3.0, 64)]:
    t0 = time.time()
    hyp = [beam_ctc(m, idx_to_char, blank, lm=lm, alfa=alfa, beta=beta, prefijos=pre, B=B) for m in mats_pm]
    print(alfa, beta, B, "CER {:.1%} | WER {:.1%}".format(*cer_wer(refs_pm, hyp)), f"| {time.time() - t0:.0f} s")

0.4 3.0 32 CER 67.1% | WER 102.4% | 16 s
0.3 2.0 32 CER 67.5% | WER 99.9% | 15 s
0.4 2.5 32 CER 68.4% | WER 98.5% | 14 s
0.4 3.0 64 CER 66.8% | WER 105.0% | 30 s


TEST: alfa:0.3, beta:2.0 y B:32

In [ ]:
# ===== TEST: una sola vez, configuración fijada antes de mirar =====
ALFA, BETA, B = 0.3, 2.0, 32

df_test = pd.read_csv(data / "test_predictions.csv").fillna("")
lp_test = np.load(data / "test_logprobs.npz")
print(len(df_test), "clips de test |", df_test["video_id"].nunique(), "videos |", len(lp_test.files), "con log_probs")

refs_t = df_test["reference"].tolist()
mats_t = [lp_test[clave(f)].astype(np.float32) for f in df_test.itertuples()]

t0 = time.time()
hyp_t = [beam_ctc(m, idx_to_char, blank, lm=lm, alfa=ALFA, beta=BETA, prefijos=pre, B=B) for m in mats_t]

print(f"\nTEST ({len(df_test)} clips) · {time.time() - t0:.0f} s")
print("argmax:      CER {:.1%} | WER {:.1%}".format(*cer_wer(refs_t, df_test["prediction"].tolist())))
print("beam + LM:   CER {:.1%} | WER {:.1%}".format(*cer_wer(refs_t, hyp_t)))

pd.DataFrame({"video_id": df_test["video_id"], "reference": refs_t,
              "argmax": df_test["prediction"], "beam": hyp_t}) \
  .to_csv(data / "test_beam.csv", index=False, encoding="utf-8")

for r, a, h in list(zip(refs_t, df_test["prediction"], hyp_t))[:5]:
    print(f"\nREAL: {r}\nARGM: {a}\nBEAM: {h}")

366 clips de test | 8 videos | 366 con log_probs

TEST (366 clips) · 46 s
argmax:      CER 68.9% | WER 100.4%
beam + LM:   CER 70.0% | WER 98.9%

REAL: la ficción es una herramienta poderosa
ARGM: a mee ua a meeta moo oa
BEAM: y a mesa a la mesa poe

REAL: una tecnología blanda
ARGM: adeo pata
BEAM: y a ceo pata

REAL: las tecnologías blandas funcionan de la misma manera
ARGM: emeo e patoe e ta memima
BEAM: a eso es pato esa meme

REAL: cuando en las historias de la biblia se nos cuenta que hay cosas que son correctas y hay cosas que son incorrectas
ARGM: o aoe te mom o ata a etos eteo e eaa o e eo eo aeta
BEAM: o a esa meme o a eso ese es a ese o eso ese

REAL: que las cosas buenas son recompensadas y las cosas malas son castigadas
ARGM: eoe poea o petetao e peto etaa
BEAM: eso es poeta compete pato esa
